In [1]:
!pip install -q pandas scikit-learn openpyxl joblib matplotlib

In [2]:
import pandas as pd
df = pd.read_csv("gallstone.csv")
df.head()

,Gallstone Status,Age,Gender,Comorbidity,Coronary Artery Disease (CAD),Hypothyroidism,Hyperlipidemia,Diabetes Mellitus (DM),Height,Weight,...,High Density Lipoprotein (HDL),Triglyceride,Aspartat Aminotransferaz (AST),Alanin Aminotransferaz (ALT),Alkaline Phosphatase (ALP),Creatinine,Glomerular Filtration Rate (GFR),C-Reactive Protein (CRP),Hemoglobin (HGB),Vitamin D
0,0,50,0,0,0,0,0,0,185,92.8,...,40.0,134.0,20.0,22.0,87.0,0.82,112.47,0.0,16.0,33.0
1,0,47,0,1,0,0,0,0,176,94.5,...,43.0,103.0,14.0,13.0,46.0,0.87,107.10,0.0,14.4,25.0
2,0,61,0,0,0,0,0,0,171,91.1,...,43.0,69.0,18.0,14.0,66.0,1.25,65.51,0.0,16.2,30.2
3,0,41,0,0,0,0,0,0,168,67.7,...,59.0,53.0,20.0,12.0,34.0,1.02,94.10,0.0,15.4,35.4
4,0,42,0,0,0,0,0,0,178,89.6,...,30.0,326.0,27.0,54.0,71.0,0.82,112.47,0.0,16.8,40.6


In [3]:
df.columns

Index(['Gallstone Status', 'Age', 'Gender', 'Comorbidity',
       'Coronary Artery Disease (CAD)', 'Hypothyroidism', 'Hyperlipidemia',
       'Diabetes Mellitus (DM)', 'Height', 'Weight', 'Body Mass Index (BMI)',
       'Total Body Water (TBW)', 'Extracellular Water (ECW)',
       'Intracellular Water (ICW)',
       'Extracellular Fluid/Total Body Water (ECF/TBW)',
       'Total Body Fat Ratio (TBFR) (%)', 'Lean Mass (LM) (%)',
       'Body Protein Content (Protein) (%)', 'Visceral Fat Rating (VFR)',
       'Bone Mass (BM)', 'Muscle Mass (MM)', 'Obesity (%)',
       'Total Fat Content (TFC)', 'Visceral Fat Area (VFA)',
       'Visceral Muscle Area (VMA) (Kg)', 'Hepatic Fat Accumulation (HFA)',
       'Glucose', 'Total Cholesterol (TC)', 'Low Density Lipoprotein (LDL)',
       'High Density Lipoprotein (HDL)', 'Triglyceride',
       'Aspartat Aminotransferaz (AST)', 'Alanin Aminotransferaz (ALT)',
       'Alkaline Phosphatase (ALP)', 'Creatinine',
       'Glomerular Filtration Rate (GFR

In [4]:
TARGET = "Visceral Fat Rating (VFR)"
if TARGET not in df.columns:
    raise ValueError(f"Target column '{TARGET}' not found.")
X = df.drop(columns=[TARGET])
y = df[TARGET]
X.head(), y.head()


(   Gallstone Status  Age  Gender  Comorbidity  Coronary Artery Disease (CAD)  \
 0                 0   50       0            0                              0   
 1                 0   47       0            1                              0   
 2                 0   61       0            0                              0   
 3                 0   41       0            0                              0   
 4                 0   42       0            0                              0   
 
    Hypothyroidism  Hyperlipidemia  Diabetes Mellitus (DM)  Height  Weight  \
 0               0               0                       0     185    92.8   
 1               0               0                       0     176    94.5   
 2               0               0                       0     171    91.1   
 3               0               0                       0     168    67.7   
 4               0               0                       0     178    89.6   
 
    ...  High Density Lipoprotein (HDL)  T

In [5]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train.shape, X_test.shape

((255, 38), (64, 38))

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
numeric_features = X.select_dtypes(include=['number']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

In [7]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
models = {
    "LinearRegression": Pipeline([('preprocessor', preprocessor),
                                  ('model', LinearRegression())]),
    "RandomForest": Pipeline([('preprocessor', preprocessor),
                              ('model', RandomForestRegressor(n_estimators=200,
                                                              random_state=42))]),
    "SVR": Pipeline([('preprocessor', preprocessor),
                     ('model', SVR())])
}

In [8]:
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error
results = []
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))
def mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    model_rmse = rmse(y_test, preds)
    model_mae = mean_absolute_error(y_test, preds)
    model_mape = mape(y_test, preds)
    results.append((name, model_rmse, model_mae, model_mape))
results

[('LinearRegression',
  np.float64(0.9992593843942189),
  0.7269151022637138,
  np.float64(11.575633419301385)),
 ('RandomForest',
  np.float64(1.4662665173835212),
  1.0409375,
  np.float64(15.994982451295641)),
 ('SVR',
  np.float64(1.7462998265251295),
  1.1120646419066038,
  np.float64(19.31924350721874))]